In [4]:
import json
import shutil
from pathlib import Path
from collections import defaultdict

In [ ]:
def move_duplicate_runs(results_dir: Path) -> None:
    """
    Identifies and moves duplicate experiment runs within a results directory.

    A run is a duplicate if its 'config.json' is identical to another run's,
    excluding the 'output_file_name' and 'timestamp' fields. For each set of
    duplicates, the most recent run is kept, and the older ones are moved.

    Args:
        results_dir: The Path object pointing to a specific dataset's results
                     folder (e.g., '../results/').
    """
    # The destination for duplicates is in the same folder as the results folder
    duplicate_dest_dir = results_dir / 'duplicate_runs'
    
    # Create the destination directory if it doesn't exist
    duplicate_dest_dir.mkdir(parents=True, exist_ok=True)
    print(f"Duplicate runs will be moved to: {duplicate_dest_dir}\n")

    if not results_dir.is_dir():
        print(f"Error: Directory not found at '{results_dir}'")
        return

    # Iterate through each model's result folder (e.g., 'llama2-7b', 'mistral', etc.)
    for model_dir in results_dir.iterdir():
        if not model_dir.is_dir():
            continue

        model_name = model_dir.name
        print(f"Processing model: {model_name}")

        runs_for_model = []
        # First, collect all valid runs for this model
        for run_dir in model_dir.iterdir():
            if not run_dir.is_dir():
                continue

            config_path = run_dir / 'config.json'
            results_path = run_dir / 'results.csv'

            # A run is only valid if it has both a config and results file
            if not (config_path.exists() and results_path.exists()):
                continue

            try:
                with open(config_path, 'r') as f:
                    config_data = json.load(f)
                
                # Create a "fingerprint" of the config by removing volatile keys
                # and creating a sorted, compact JSON string. This allows us
                # to compare configurations for equality.
                key_config = config_data.copy()
                key_config.pop('output_file_name', None)
                key_config.pop('timestamp', None)
                
                fingerprint = json.dumps(key_config, sort_keys=True)
                
                runs_for_model.append({
                    'path': run_dir,
                    'timestamp': run_dir.name, # The folder name is the timestamp
                    'fingerprint': fingerprint
                })

            except json.JSONDecodeError as e:
                print(f"  - Error parsing config.json in {run_dir}: {e}")

        # Now, group the collected runs by their fingerprint
        grouped_runs = defaultdict(list)
        for run in runs_for_model:
            grouped_runs[run['fingerprint']].append(run)
            
        # Process the groups to find and move duplicates
        for fingerprint, run_group in grouped_runs.items():
            if len(run_group) > 1:
                print(f"  - Found {len(run_group)} duplicate runs for a configuration.")
                
                # Sort runs by timestamp, newest first
                run_group.sort(key=lambda r: r['timestamp'], reverse=True)
                
                # The first one is the latest, which we will keep
                run_to_keep = run_group[0]
                print(f"    - Keeping latest: {run_to_keep['path'].name}")
                
                # The rest are older duplicates that need to be moved
                runs_to_move = run_group[1:]
                for run in runs_to_move:
                    source_path = run['path']
                    dest_path = duplicate_dest_dir / source_path.name
                    print(f"    - Moving older: {source_path.name}")
                    try:
                        shutil.move(str(source_path), str(dest_path))
                    except Exception as e:
                        print(f"      ! Error moving {source_path}: {e}")
        print("-" * 20)

In [6]:
# You can process one or multiple dataset folders
results_folder = Path('../results')

# Run the function to find and move duplicates
move_duplicate_runs(results_folder)

print("\nDuplicate cleanup process finished.")

Duplicate runs will be moved to: ../results/duplicate_runs

Processing model: Qwen3-4B
--------------------
Processing model: Qwen3-4B-Base
--------------------
Processing model: duplicate_runs
--------------------

Duplicate cleanup process finished.
